In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent.parent.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

from ocr_vs_vlm.results_final.shared.colors import get_model_color, APPROACH_COLORS
from ocr_vs_vlm.results_final.shared.stats_utils import bootstrap_ci, wilcoxon_test, cohens_d
from ocr_vs_vlm.results_final.shared.viz_utils import setup_plotly_template, create_metric_boxplot, create_radar_chart
from ocr_vs_vlm.results_final.shared.data_loader import load_dataset_data, extract_models_from_columns, PHASE_TO_APPROACH

setup_plotly_template()
print("✓ Setup complete")

✓ Setup complete


## 1. Load IAM Mini Data

In [2]:
DATASET = 'IAM_mini'

try:
    df = load_dataset_data(DATASET)
    print(f"✓ Loaded {len(df)} samples")
    print(f"  Phases: {df['phase'].unique().tolist()}")
except Exception as e:
    print(f"Error: {e}")
    df = pd.DataFrame()

models = extract_models_from_columns(df) if len(df) > 0 else []
print(f"  Models: {models}")

✓ Loaded 1500 samples
  Phases: ['phase_1', 'phase_3', 'phase_2']
  Models: ['azure_intelligence', 'claude_sonnet', 'gpt-5-mini', 'gpt-5-nano', 'mistral_document_ai']


## 2. CER/WER Analysis

In [3]:
# Extract metrics
metrics_data = []
for phase in df['phase'].unique() if len(df) > 0 else []:
    phase_df = df[df['phase'] == phase]
    approach = PHASE_TO_APPROACH.get(phase, phase)
    
    for model in models:
        for metric in ['cer', 'wer', 'anls']:
            col = f'{metric}_{model}'
            if col in phase_df.columns:
                scores = phase_df[col].dropna().values
                if len(scores) > 0:
                    mean, ci_lo, ci_hi = bootstrap_ci(scores, 'mean')
                    metrics_data.append({
                        'Phase': phase,
                        'Approach': approach,
                        'Model': model,
                        'Metric': metric.upper(),
                        'Value': mean,
                        'CI_Lower': ci_lo,
                        'CI_Upper': ci_hi
                    })

metrics_df = pd.DataFrame(metrics_data)
if len(metrics_df) > 0:
    pivot = metrics_df.pivot_table(index=['Phase', 'Model'], columns='Metric', values='Value')
    display(pivot.round(4))

In [4]:
# Dual-axis plot: CER and WER
if len(metrics_df) > 0:
    fig = make_subplots(rows=1, cols=2, subplot_titles=['Character Error Rate (CER)', 'Word Error Rate (WER)'])
    
    for i, metric in enumerate(['CER', 'WER']):
        metric_data = metrics_df[metrics_df['Metric'] == metric]
        for model in metric_data['Model'].unique():
            m_data = metric_data[metric_data['Model'] == model]
            fig.add_trace(
                go.Bar(x=m_data['Phase'], y=m_data['Value'], name=model if i == 0 else None,
                      marker_color=get_model_color(model), showlegend=(i == 0)),
                row=1, col=i+1
            )
    
    fig.update_layout(title='IAM Mini: OCR vs VLM for Handwriting Recognition', height=450, barmode='group')
    fig.show()

## 3. OCR vs VLM Comparison

In [5]:
# Statistical comparison
if len(metrics_df) > 0:
    print("📊 OCR (P-OCR) vs VLM (P-VLM_simple/taskaware) - CER Comparison")
    print("-" * 60)
    
    ocr_data = metrics_df[(metrics_df['Phase'] == 'P-OCR') & (metrics_df['Metric'] == 'CER')]
    vlm_data = metrics_df[(metrics_df['Phase'].isin(['P-VLM_simple', 'P-VLM_taskaware'])) & (metrics_df['Metric'] == 'CER')]
    
    if len(ocr_data) > 0 and len(vlm_data) > 0:
        ocr_mean = ocr_data['Value'].mean()
        vlm_mean = vlm_data['Value'].mean()
        
        print(f"OCR Average CER: {ocr_mean:.4f}")
        print(f"VLM Average CER: {vlm_mean:.4f}")
        print(f"Winner: {'OCR' if ocr_mean < vlm_mean else 'VLM'} (lower is better)")

## 4. Multi-Metric Radar

In [6]:
# Radar chart comparing models across metrics
if len(metrics_df) > 0:
    # Prepare data for radar (normalize: invert CER/WER so higher=better)
    radar_data = {}
    for model in models:
        m_data = metrics_df[metrics_df['Model'] == model].groupby('Metric')['Value'].mean()
        if len(m_data) > 0:
            radar_data[model] = {
                'CER (inv)': 1 - m_data.get('CER', 0.5),  # Invert: higher=better
                'WER (inv)': 1 - m_data.get('WER', 0.5),
                'ANLS': m_data.get('ANLS', 0.5)
            }
    
    if radar_data:
        fig = create_radar_chart(radar_data, ['CER (inv)', 'WER (inv)', 'ANLS'], 
                                title='IAM Mini: Model Comparison (Higher=Better)')
        fig.show()

## 5. Key Findings

In [7]:
print("=" * 70)
print("IAM MINI - KEY FINDINGS")
print("=" * 70)

if len(metrics_df) > 0:
    cer_data = metrics_df[metrics_df['Metric'] == 'CER']
    if len(cer_data) > 0:
        best = cer_data.nsmallest(1, 'Value').iloc[0]
        print(f"\n🏆 Best CER: {best['Model']} ({best['Phase']}) → CER={best['Value']:.4f}")
    
    print("\n📊 By Phase:")
    for phase in cer_data['Phase'].unique():
        phase_mean = cer_data[cer_data['Phase'] == phase]['Value'].mean()
        print(f"   {phase}: avg CER = {phase_mean:.4f}")

print("\n" + "=" * 70)

IAM MINI - KEY FINDINGS

